# pydantic格式的使用

## 1 基本使用

举例：

In [19]:
from langchain_deepseek import ChatDeepSeek
import json

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from rich import print as rprint
from config.config import DeepSeekModel


model = ChatDeepSeek(
    base_url=DeepSeekModel.deepseek_base_url,
    api_key=DeepSeekModel.deepseek_api_key,
    model=DeepSeekModel.deepseek_model,
    extra_body={"thinking": {"type": "disabled"}} # 核心：关闭思考模式
)

In [3]:
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 构造Prompt，强制输出JSON
prompt = """
提取人物信息，严格输出JSON，不要多余文字、markdown。
schema：{{"name":"string","age":"integer","occupation":"string"}}

输入：{input_text}
"""

resp = model.invoke(prompt.format(input_text="张三是一名30岁的软件工程师"))
# 解析字符串到Pydantic对象
raw_json = json.loads(resp.content)

result = Person(**raw_json)

print(result)
print(type(result))
# Person(name='张三', age=30, occupation='软件工程师')

name='张三' age=30 occupation='软件工程师'
<class '__main__.Person'>


In [15]:
class Redio(BaseModel):
    """电影信息"""
    name: str = Field(description="电影名字")
    year: int = Field(description="上映年份")
    lead : str = Field(description="主演")


prompt = ChatPromptTemplate.from_messages([
    ("""
        提取电影信息，严格输出JSON，不要多余文字、markdown。
        schema：{{"name":"string","year":"integer","lead":"string"}}

        输入：{input_text}
    """)
])

response = model.invoke(prompt.format(input_text="请列出钢铁侠一的信息"))


result = Redio.model_validate_json(response.content)

rprint(result)

Redio(name='钢铁侠', year=2008, lead='小罗伯特·唐尼')

In [20]:
class Redio(BaseModel):
    """电影信息"""
    name: str = Field(description="电影名字")
    year: int = Field(description="上映年份")
    lead : str = Field(description="主演")

redio_output_model = model.with_structured_output(Redio)

response = redio_output_model.invoke("钢铁侠1电影")

rprint(response)

Redio(name='钢铁侠', year=2008, lead='小罗伯特·唐尼')